In [4]:
## 경고 제거##
import warnings
warnings.filterwarnings('always')
warnings.filterwarnings('ignore')

## DataFrame & visual option ##
import pandas as pd
from IPython.core.display import display, HTML
display(HTML("<style>.container { width:100% !important; }</style>"))
pd.set_option('display.max_columns', None)
import polars as pl

import numpy as np
import os
import json

import re
## geo ##
import geopandas as gpd

#progress bar
from tqdm.auto import tqdm

## Visualize option ##
import seaborn as sns
import matplotlib.pyplot as plt
import ipywidgets as widgets

plt.rcParams['axes.unicode_minus'] = False
plt.rcParams['font.family'] = 'D2Coding'

# MTS
- 총 3,226개
    - 제어기 위/경도가 없는 장비 688개 제외
    - 2,538건으로 진행
- 관리_설치회차가 2회 이상인 경우 9개 존재
    - 해당 장비의 경우 설치일자가 관리_설치일과 다름
    - 설치일 기준은 "설치일자" 컬럼을 사용
- 구간과속카메라 중 수원 금곡동에 설치된 카메라는 시/종점 위경도 좌표가 동일하여 제외

In [134]:
mts = pd.read_csv('./tohome/230920_교통과학장비 설치정보_경기남부.csv')

In [15]:
mts.shape

(3226, 40)

In [16]:
print(mts[mts['제어기 경도'].isna()].shape)
print(mts[mts['제어기 위도'].isna()].shape)

(688, 40)
(688, 40)


In [17]:
mts = mts[~mts['제어기 경도'].isna()]
#컬럼명 '*' 제거
mts.columns = [x.replace('*','') for x in mts.columns]

In [18]:
mts.shape

(2538, 40)

In [19]:
mts['설치일자'] = pd.to_datetime(mts['설치일자'])
mts['관리·설치일'] = pd.to_datetime(mts['관리·설치일'])

In [20]:
mts[mts['설치일자'] != mts['관리·설치일']]

,장비번호,장비종류,단속형태,납품업체,제조업체,제조번호,최초운영일자,장비규격,구간그룹명,지방청,자산구분,관리·설치일,설치구분/지자체명,관리종료일,종료구분,관리이관,설치(이전)회차,설치일자,설치장소,센터와의거리,관할서,제어기 위도,제어기 경도,표지판 위도,표지판 경도,표지판 개수,차량검지,설치·단속 차로수,도로차로수(편도),제한속도(소형),제한속도(대형),단속속도(소형),단속속도(대형),도로종류,노선번호,노선명,특정도로,도로유형,도로선형,조명장치
16,F8998,신호위반,신호위반,진우산전,진우산전,NaN,2016-05-12,('12.03) 2차 표준화,NaN,경기남부,경찰청,2016-05-12,NaN,NaN,NaN,NaN,2,2019-07-09,화성시 향남읍 관리5-1 동오사거리(정남→백토리산업단지),26.7,화성서부경찰서,37.130986,126.952939,37.139673,126.956319,2.0,루프센서,2.0,2.0,60,60,71.0,71.0,지방도,309,화성시 향남읍,NaN,교차로,직선내리막,NaN
119,F9867,속도위반,속도위반,휴앤에스,휴앤에스,NaN,2017-04-20,('12.03) 2차 표준화,NaN,경기남부,경찰청,2017-04-20,NaN,NaN,NaN,NaN,2,2019-07-09,광명시 소하동 548-4 소하지하차도 전 150m지점(광명시청->광명역),NaN,NaN,37.442093,126.894924,37.448873,126.890423,NaN,루프센서,NaN,NaN,50,50,NaN,NaN,NaN,NaN,지방도(하안로),NaN,NaN,NaN,NaN
151,F9899,신호위반,신호위반,휴앤에스,휴앤에스,NaN,2017-04-20,('12.03) 2차 표준화,NaN,경기남부,경찰청,2017-04-20,NaN,NaN,NaN,NaN,2,2020-09-10,하남시 초이동 65-9 초이동화훼단지앞(서울→상일IC),NaN,NaN,37.544225,127.167285,37.540035,127.157221,NaN,루프센서,NaN,NaN,60,60,NaN,NaN,NaN,NaN,국도43(천호대로),NaN,NaN,NaN,NaN
158,F9906,신호위반,신호위반,휴앤에스,휴앤에스,NaN,2017-04-20,('12.03) 2차 표준화,NaN,경기남부,경찰청,2017-04-20,NaN,NaN,NaN,NaN,2,2021-02-27,화성시 비봉면 양노리 632-3 양노3리버스정류장 앞(송산마도IC→비봉IC),NaN,NaN,37.226563,126.857754,37.223077,126.850003,NaN,루프센서,NaN,NaN,70,70,NaN,NaN,NaN,NaN,지방도313,NaN,NaN,NaN,NaN
191,G0054,속도위반,속도위반,진우산전,진우산전,NaN,2017-06-30,('12.03) 2차 표준화,NaN,경기남부,지자체,2017-06-30,용인시,NaN,NaN,NaN,2,2020-06-29,용인시 처인구 남사면 봉명리 199-1 유평1교차로 800m지점(장지IC->남사IC),NaN,용인동부경찰서,37.118313,127.123391,37.129697,127.124721,2.0,루프센서,2.0,2.0,80,80,NaN,NaN,지방도,23,국지도23(용구대로),NaN,단일로,직선평지,NaN
220,G0139,신호위반,신호위반,토페스,토페스,NaN,2017-07-27,('12.03) 2차 표준화,NaN,경기남부,지자체,2017-07-27,시흥시,NaN,NaN,NaN,2,2020-09-27,시흥시 정왕동 1222 시흥정왕동우체국 앞 사거리(정왕동체육공원->시흥소방서),NaN,시흥경찰서,37.349795,126.739485,37.354426,126.731803,2.0,루프센서,2.0,4.0,50,50,61.0,61.0,국도,NaN,지방도(정왕신길로),NaN,교차로,직선평지,NaN
221,G0140,신호위반,신호위반,토페스,토페스,NaN,2017-07-27,('12.03) 2차 표준화,NaN,경기남부,지자체,2017-07-27,시흥시,NaN,NaN,NaN,2,2020-09-26,시흥시 정왕신길로 206(정왕동 1800-1) 소방서사거리(배곧동→정왕동체육공원),NaN,시흥경찰서,37.344878,126.747842,37.340048,126.756065,2.0,루프센서,2.0,4.0,50,50,61.0,61.0,국도,NaN,지방도(정왕신길로),NaN,교차로,직선평지,NaN
381,G0983,속도위반,속도위반,진우산전,진우산전,NaN,2018-02-13,('17.03) -아,NaN,경기남부,경찰청,2018-02-13,NaN,NaN,NaN,NaN,2,2020-06-14,하남시 덕풍동 34 신풍지하차도 지나 150m지점(팔당대교->미사IC),NaN,하남경찰서,37.554112,127.210962,37.552986,127.212011,2.0,루프센서,2.0,3.0,80,80,NaN,NaN,시도,NaN,NaN,NaN,단일로,직선평지,NaN
498,G1425,신호위반,신호위반,동아신호,건아정보기술,NaN,2018-07-30,('12.03) 2차 표준화,NaN,경기남부,지자체,2018-07-30,광주시,NaN,NaN,NaN,2,2020-11-25,광주시 역동 216-216 경기광주역 앞 교차로(광주시청->용인),NaN,NaN,37.398351,127.254002,37.404600,127.261306,NaN,루프센서,NaN,NaN,60,60,NaN,NaN,NaN,NaN,NaN,NaN,단일로,NaN,NaN


## 2021년 설치한 장비만 추출
- 447개

In [21]:
Equ_est21 = mts[mts['설치일자'].dt.year==2021]

In [22]:
Equ_est21.columns

Index(['장비번호', '장비종류', '단속형태', '납품업체', '제조업체', '제조번호', '최초운영일자', '장비규격',
       '구간그룹명', '지방청', '자산구분', '관리·설치일', '설치구분/지자체명', '관리종료일', '종료구분', '관리이관',
       '설치(이전)회차', '설치일자', '설치장소', '센터와의거리', '관할서', '제어기 위도', '제어기 경도',
       '표지판 위도', '표지판 경도', '표지판 개수', '차량검지', '설치·단속 차로수', '도로차로수(편도)',
       '제한속도(소형)', '제한속도(대형)', '단속속도(소형)', '단속속도(대형)', '도로종류', '노선번호', '노선명',
       '특정도로', '도로유형', '도로선형', '조명장치'],
      dtype='object')

In [23]:
Equ_est21 = Equ_est21[['장비번호', '장비종류', '단속형태','최초운영일자','구간그룹명',
           '관리·설치일','설치(이전)회차','설치일자', '설치장소','제어기 위도', '제어기 경도',
            '표지판 개수', '차량검지','도로유형',]]

In [26]:
Equ_est21

,장비번호,장비종류,단속형태,최초운영일자,구간그룹명,관리·설치일,설치(이전)회차,설치일자,설치장소,제어기 위도,제어기 경도,표지판 개수,차량검지,도로유형
158,F9906,신호위반,신호위반,2017-04-20,NaN,2017-04-20,2,2021-02-27,화성시 비봉면 양노리 632-3 양노3리버스정류장 앞(송산마도IC→비봉IC),37.226563,126.857754,NaN,루프센서,NaN
1324,G6950,속도위반,속도위반,2021-01-31,NaN,2021-01-31,1,2021-01-31,여주시 능서면 오계리 422-6 오계2리 버스정류장 부근(광대리→여주컨트리클럽),37.272716,127.588824,2.0,레이다센서,단일로
1325,G6951,속도위반,속도위반,2021-01-31,NaN,2021-01-31,1,2021-01-31,여주시 대신면 초현리 17-1 초현1리마을입구(양평 → 대신면),37.385818,127.600621,2.0,레이다센서,단일로
1326,G6952,속도위반,속도위반,2021-01-31,NaN,2021-01-31,1,2021-01-31,안산시 단원구 원선로 65 안산서초교 앞 교차로 어린이보호구역(선부광장→안산역),37.330202,126.799221,2.0,레이다센서,교차로
1327,G6953,속도위반,속도위반,2021-01-31,NaN,2021-01-31,1,2021-01-31,군포시 산본동 1087-4 한양목련(아) 1210동 앞 삼거리 어린이보호구역(태을초...,37.371599,126.928310,2.0,레이다센서,교차로
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1765,H1438,신호위반,신호위반,2021-12-30,NaN,2021-12-30,1,2021-12-30,화성시 산척동 780 세정초교 앞 사거리 어린이보호구역(반도유보라 → 세정초교),37.167670,127.125129,2.0,레이다센서,교차로
1766,H1439,신호위반,신호위반,2021-12-30,NaN,2021-12-30,1,2021-12-30,화성시 산척동 765-3 정현초교 정문 앞 어린이보호구역(동탄산척로 → 부영2485동),37.171713,127.114003,2.0,레이다센서,교차로
1767,H1440,신호위반,신호위반,2021-12-30,NaN,2021-12-30,1,2021-12-30,화성시 산척동 765-3 정현초교 정문 앞 어린이보호구역(부영2485동 → 동탄산척로),37.171713,127.114003,2.0,레이다센서,교차로
1768,H1441,신호위반,신호위반,2021-12-30,NaN,2021-12-30,1,2021-12-30,화성시 장지동 1036 화남초교 정문 앞 어린이보호구역(동탄호수공원 → 동탄28단지),37.162945,127.096745,2.0,레이다센서,교차로


In [27]:
Equ_est21.shape

(447, 14)

## 구간과속
- LINK와 카메라를 연결하기 위해서는 구간그룹명으로 연결해야 함
- buffer는 QGIS 에서 제작
- 범위는 20m

In [23]:
tmp = gpd.read_file('./use_data/gis/SLTD_SECTION_LINK_5179.shp')

In [24]:
tmp = tmp[tmp['LINK_ID'] != '2140018400']

In [25]:
sltd = pd.read_csv('./구간과속카메라_구간_LINK_ID저장.csv')

In [26]:
sltd

,장비번호,구간명,LINK_ID
0,G3758,김포시 운양동(강화→서울),2320240300
1,G3758,김포시 운양동(강화→서울),2320262300
2,G3758,김포시 운양동(강화→서울),2320282000
3,G3758,김포시 운양동(강화→서울),2320132500
4,G3758,김포시 운양동(강화→서울),2320128200
...,...,...,...
151,G1915,성남-장호원간(광주->부발IC),2300539300
152,G1915,성남-장호원간(광주->부발IC),2300405601
153,G1915,성남-장호원간(광주->부발IC),2300405602
154,G1915,성남-장호원간(광주->부발IC),2300405800


In [27]:
tmp = tmp[['LINK_ID','geometry']]

In [28]:
tmp['LINK_ID'] = tmp['LINK_ID'].astype('str')
sltd['LINK_ID'] = sltd['LINK_ID'].astype('str')

In [29]:
tmp['LINK_ID'] = tmp['LINK_ID'].astype('str')
sltd['LINK_ID'] = sltd['LINK_ID'].astype('str')

In [30]:
sltd[sltd['LINK_ID'] == tmp['LINK_ID'][0]].iloc[0,1]

'성남장호원간'

In [31]:
num_list = []
nm_list = [] 
for i in tmp['LINK_ID']:
    equ_num = sltd[sltd['LINK_ID'] == i].iloc[0,0]
    sec_nm = sltd[sltd['LINK_ID'] == i].iloc[0,1]
    num_list.append(equ_num)
    nm_list.append(sec_nm)

tmp['EQU_NUM'] = num_list
tmp['SECTION_NM'] = nm_list
    
    
    

In [27]:
tmp.to_file('./use_data/gis/section_polygon/test.shp',driver = 'ESRI Shapefile', encoding='cp949')

In [32]:
from shapely.geometry import LineString, MultiLineString
from shapely.ops import linemerge

In [33]:
multiline_geometries = []
site_list = []
for site in tmp['SECTION_NM'].unique():
    multi_line = MultiLineString(tmp[tmp['SECTION_NM']==site]['geometry'].to_list())
    multiline_geometries.append(multi_line)
    site_list.append(site)

# 새로운 GeoDataFrame 생성
new_data = {"SECTION_NM": site_list, "geometry": multiline_geometries}
new_gdf = gpd.GeoDataFrame(new_data)

# 결과 확인
display(new_gdf)

,SECTION_NM,geometry
0,성남장호원간,"MULTILINESTRING ((984547.917 1930276.096, 9849..."
1,시흥 도심형구간(119안전센터->은계중학교),"MULTILINESTRING ((937466.032 1938012.668, 9374..."
2,서해안302,"MULTILINESTRING ((945123.947 1915769.540, 9450..."
3,"오성면구간,신대리구간","MULTILINESTRING ((954771.018 1882765.479, 9547..."
4,경부하행,"MULTILINESTRING ((966021.591 1900538.546, 9660..."
5,성남-장호원간(이천->성남),"MULTILINESTRING ((982148.982 1931176.260, 9821..."
6,성남-장호원간(광주->부발IC),"MULTILINESTRING ((1000496.389 1922654.990, 100..."
7,경부고속도로390,"MULTILINESTRING ((964880.407 1918741.302, 9648..."
8,영동고속도로,"MULTILINESTRING ((971918.359 1919140.227, 9722..."
9,김포 고촌읍 신곡리(서울->강화),"MULTILINESTRING ((936164.328 1956957.595, 9361..."


In [34]:
new_gdf.to_file('./use_data/gis/section_polygon/multi_line.shp',driver = 'ESRI Shapefile', encoding='cp949')

# TAAS

## 경기도로 분할

In [35]:
taas = pd.read_csv('../../../011.데이터/01.TAAS/TAAS데이터/taas001h_2018_2022.csv')

In [41]:
gg_taas = taas[taas['CD_003'].between(1300,1340)]

In [42]:
gg_taas.to_csv('../../../011.데이터/01.TAAS/TAAS데이터/gg_taas.csv',index=False,encoding='utf8')

## 경기도 TAAS

In [3]:
gg_taas = pd.read_csv('../../../011.데이터/01.TAAS/TAAS데이터/gg_taas.csv')
taas_cd = pd.read_excel('../../../011.데이터/01.TAAS/TAAS데이터/tass_code.xlsx')

gg_taas.drop(columns = ['YN_133'],inplace=True)
taas_col = {key : value for key,value in zip(taas_cd['영문컬럼'],taas_cd['컬럼명'])}
gg_taas.columns = [taas_col.get(x) for x in gg_taas.columns]

#### 지역코드

data = [
    ("가평군", 1322),("고양시", 1318), ("과천시", 1332), ("광명시", 1309), ("광주시", 1319), ("구리시", 1310), ("군포시", 1333), ("김포시", 1327),
    ("남양주시", 1334), ("동두천시", 1330), ("부천시", 1306), ("성남시", 1303), ("수원시", 1302), ("시흥시", 1316), ("안산시", 1307), ("안성시", 1326), ("안양시", 1305),
    ("양주시", 1311),("양평군", 1323),("여주시", 1313),("연천군", 1320), ("오산시", 1335), ("용인시", 1325), ("의왕시", 1336), ("의정부시", 1304), ("이천시", 1324),
    ("파주시", 1317),("평택시", 1308), ("포천시", 1321), ("하남시", 1337), ("화성시", 1315)
]

taas_sgg_dict = {key : value for value,key in data}



# 변환 완료 데이터셋(TAAS 기반)
- TAAS에 MTS와 연결할 수 있는 key 정보를 담아놓음
    - 신호, 과속 단속 장비 : 장비번호
    - 구간과속 단속 장비 : 구간그룹명
> 구간과속의 경우 장비번호로 묶기는 어려움
  - 구간단속의 경우 한 도로안의 여러개의 카메라를 구간안에 묶었기 때문에 다른 장비의 경우에도 동일하게 구간그룹명과 같은 부분을 넣어주어야 함

In [29]:
light = gpd.read_file('./use_data/gis/MTS_LIGHT_SLTD_ACC_4326.shp',encoding='cp949')
print('END -- READ LIGHT')
speed = gpd.read_file('./use_data/gis/MTS_SPEED_SLTD_ACC_4326.shp',encoding='cp949')
print('END -- READ SPEED')
section = gpd.read_file('./use_data/gis/MTS_SECTION_SLTD_ACC_4326.shp',encoding='cp949')
print('END -- READ SECTION')

END -- READ LIGHT
END -- READ SPEED
END -- READ SECTION


In [31]:
taas_cd = pd.read_excel('../../../011.데이터/01.TAAS/TAAS데이터/tass_code.xlsx')
taas_col = {key : value for key,value in zip(taas_cd['영문컬럼'],taas_cd['컬럼명'])}

data = [
    ("가평군", 1322),("고양시", 1318), ("과천시", 1332), ("광명시", 1309), ("광주시", 1319), ("구리시", 1310), ("군포시", 1333), ("김포시", 1327),
    ("남양주시", 1334), ("동두천시", 1330), ("부천시", 1306), ("성남시", 1303), ("수원시", 1302), ("시흥시", 1316), ("안산시", 1307), ("안성시", 1326), ("안양시", 1305),
    ("양주시", 1311),("양평군", 1323),("여주시", 1313),("연천군", 1320), ("오산시", 1335), ("용인시", 1325), ("의왕시", 1336), ("의정부시", 1304), ("이천시", 1324),
    ("파주시", 1317),("평택시", 1308), ("포천시", 1321), ("하남시", 1337), ("화성시", 1315)
]

taas_sgg_dict = {key : value for value,key in data}

light_df = light.filter(regex=re.compile(r'^(?!CD_016_L|CD_044_L|CD_052_L|CD_027_1_|CD_027_2_|YN_133)'))

light_df.columns = [taas_col.get(x) for x in light_df.filter(regex=re.compile(r'^(?!CD_016_L|CD_044_L|CD_052_L|CD_027_1_|CD_027_2_|YN_133)')).columns]
light_df_chg_colist = light_df.columns.to_list()
light_df_chg_colist[-2] = '장비번호'
light_df_chg_colist[-1] = 'geometry'
light_df.columns = light_df_chg_colist

speed_df = speed.filter(regex=re.compile(r'^(?!CD_016_L|CD_044_L|CD_052_L|CD_027_1_|CD_027_2_|YN_133)'))

speed_df.columns = [taas_col.get(x) for x in speed_df.filter(regex=re.compile(r'^(?!CD_016_L|CD_044_L|CD_052_L|CD_027_1_|CD_027_2_|YN_133)')).columns]
speed_df_chg_colist = speed_df.columns.to_list()
speed_df_chg_colist[-2] = '장비번호'
speed_df_chg_colist[-1] = 'geometry'
speed_df.columns = speed_df_chg_colist

section_df = section.filter(regex=re.compile(r'^(?!CD_016_L|CD_044_L|CD_052_L|CD_027_1_|CD_027_2_|YN_133)'))
section_df.columns = [taas_col.get(x) for x in section_df.filter(regex=re.compile(r'^(?!CD_016_L|CD_044_L|CD_052_L|CD_027_1_|CD_027_2_|YN_133)')).columns]
section_df_chg_colist = section_df.columns.to_list()
section_df_chg_colist[-2] = '구간그룹명'
section_df_chg_colist[-1] = 'geometry'
section_df.columns = section_df_chg_colist


light_df['발생일시'] = light_df['발생일시'].astype('str')
speed_df['발생일시'] = speed_df['발생일시'].astype('str')
section_df['발생일시'] = speed_df['발생일시'].astype('str')

def extract_ymd(x):
    return x[:8]
def extract_time(x):
    return x[8:]

light_df['발생일자'] = light_df['발생일시'].apply(lambda x : extract_ymd(x))
speed_df['발생일자'] = speed_df['발생일시'].apply(lambda x : extract_ymd(x))
section_df['발생일자'] = section_df['발생일시'].apply(lambda x : extract_ymd(x))

light_df['발생시각'] = light_df['발생일시'].apply(lambda x : extract_time(x))
speed_df['발생시각'] = speed_df['발생일시'].apply(lambda x : extract_time(x))
section_df['발생시각'] = section_df['발생일시'].apply(lambda x : extract_time(x))

light_df['발생일자'] = pd.to_datetime(light_df['발생일자'])
speed_df['발생일자'] = pd.to_datetime(speed_df['발생일자'])
section_df['발생일자'] = pd.to_datetime(section_df['발생일자'])

sltd_light_acc = light_df[light_df['발생일자'].between('2020-01-01','2022-12-31')]
sltd_speed_acc = speed_df[speed_df['발생일자'].between('2020-01-01','2022-12-31')]
sltd_section_acc = section_df[section_df['발생일자'].between('2020-01-01','2022-12-31')]





## 신호/과속 장비와 구간단속 장비로 MTS 분할
- 신호/과속 장비의 경우 장비번호로 매핑
- 구간단속 장비의 경우 구간그룹명으로 매핑
    - 구간 단속 장비의 설치일자는 구간그룹별로 동일 
    - 동일한 그룹은 모두 동일한 날짜에 설치됨
    - 그러나, TEMS와의 연결을 위해서는 단속장비의 번호가 필요함

In [28]:
Equ_pnt_21 = Equ_est21[Equ_est21['단속형태'] !='구간과속']
Equ_sec_21 = Equ_est21[Equ_est21['단속형태'] =='구간과속']

In [27]:
Equ_pnt_21.to_csv('./use_data/csv/mts_est_21_pnt.csv',index=False, encoding='cp949')
Equ_sec_21.to_csv('./use_data/csv/mts_est_21_sec.csv',index=False, encoding='cp949')

# TAAS에서 사고 내용 필요 컬럼만 취득
- 대상 : light_df, speed_df, section_df

필요 컬럼
- 수치형 변수만 선택
- 코드성 데이터는 변환하여 사용하기 어려움
    - 사고가 2회 이상인 날이 존재하는데 이러한 일자의 경우 평균을 구할 수 없음
    

In [34]:
sltd_columns = [
    '발생지시군구코드','주야','사망자수','중상자수','경상자수','부상신고자수','관련차량대수','장비번호','발생일자','발생시각'
]

sltd_sec_columns = [
    '발생지시군구코드','주야','사망자수','중상자수','경상자수','부상신고자수','관련차량대수','구간그룹명','발생일자','발생시각'
]

sltd_light_acc = sltd_light_acc[sltd_columns]
sltd_speed_acc = sltd_speed_acc[sltd_columns]
sltd_section_acc = sltd_section_acc[sltd_sec_columns]

In [ ]:
sltd_light_acc.to_csv('./taas_light_acc_sltd.csv',index=False,encoding='cp949')
sltd_speed_acc.to_csv('./taas_speed_acc_sltd.csv',index=False,encoding='cp949')
sltd_section_acc.to_csv('./taas_section_acc_sltd.csv',index=False,encoding='cp949')

In [49]:
tmp = pd.read_csv('./use_data/csv/taas_section_acc_sltd.csv',encoding='cp949')

In [52]:
tmp['구간그룹명'].unique()

array(['성남장호원간', '시흥 도심형구간(119안전센터->은계중학교)', '서해안302', '오성면구간,신대리구간',
       '경부하행', '성남-장호원간(이천->성남)', '성남-장호원간(광주->부발IC)', '경부고속도로390',
       '김포 고촌읍 신곡리(서울->강화)', '성남-장호원간(성남->이천)', '김포 고촌읍(강화->서울)',
       '김포시 운양동(강화→서울)', '서해안고속도로318'], dtype=object)

# 기상청 연결 지역코드 변환
- 단속장비 설치지역 기준
    - 이천, 광주, 여주 -> 이천 관측소
    - 양평, 구리, 남양주, 하남 -> 양평 관측소
    - 그외 지역 -> 수원

In [26]:
def get_weather_region_name(x):
    if x in ['이천시','광주시','여주시'] :
        return '이천'
    elif x in ['양평군','구리시','남양주시','하남시']:
        return '양평'
    else:
        return '수원'

Equ_pnt_21['설치시군구명'] = Equ_pnt_21['설치장소'].apply(lambda x : x[:3])
Equ_pnt_21['기상관측연결지역명'] = Equ_pnt_21['설치시군구명'].apply(lambda x: get_weather_region_name(x))

Equ_sec_21['설치시군구명'] = Equ_sec_21['설치장소'].apply(lambda x : x[:3])
Equ_sec_21['기상관측연결지역명'] = Equ_sec_21['설치시군구명'].apply(lambda x: get_weather_region_name(x))

In [157]:
sltd_light_acc['발생지시군구명'] = sltd_light_acc['발생지시군구코드'].apply(lambda x: taas_sgg_dict.get(x))
sltd_speed_acc['발생지시군구명'] = sltd_speed_acc['발생지시군구코드'].apply(lambda x: taas_sgg_dict.get(x))
sltd_section_acc['발생지시군구명'] = sltd_section_acc['발생지시군구코드'].apply(lambda x: taas_sgg_dict.get(x))

sltd_light_acc['기상관측연결지역명'] = sltd_light_acc['발생지시군구명'].apply(lambda x: get_weather_region_name(x))
sltd_speed_acc['기상관측연결지역명'] = sltd_speed_acc['발생지시군구명'].apply(lambda x: get_weather_region_name(x))
sltd_section_acc['기상관측연결지역명'] = sltd_section_acc['발생지시군구명'].apply(lambda x: get_weather_region_name(x))

# TEMS
- TEMS의 경우 장비번호별로 평균 취득 후 결합해야 함

In [2]:
tems = pd.read_csv('../../../011.데이터/02.TEMS/L_local_violation_20to23.csv',
                   usecols=['VIOLATION_NO','LOCAL_NO','RECEIVE_DATE','VIOLATION_TYPE', 'LIMIT_SPEED NULLIF (LIMIT_SPEED="NULL")',
                            'VIOLATION_SPEED NULLIF (VIOLATION_SPEED="NULL")','OVER_SPEED NULLIF (OVER_SPEED="NULL")',])


In [4]:
def extract_ymd(x):
    return x[:8]
def extract_time(x):
    return x[8:]


for col in ['VIOLATION_NO','LOCAL_NO','RECEIVE_DATE','VIOLATION_TYPE']:
    tems[col] = tems[col].apply(lambda x : x.replace('"',''))
    print(f'end {col}')

tems['단속일자'] = tems['RECEIVE_DATE'].apply(lambda x : extract_ymd(x))
tems['단속시간'] = tems['RECEIVE_DATE'].apply(lambda x : extract_time(x))

tems['단속일자'] = pd.to_datetime(tems['단속일자'])

end VIOLATION_NO
end LOCAL_NO
end RECEIVE_DATE
end VIOLATION_TYPE


In [5]:
tems = tems[tems['단속일자'].between('2020-01-01','2022-12-31')]

In [6]:
tems.columns = ['사건번호','장비번호','단속발생시간','단속유형','제한속도','과속속도','초과속도','단속일자','단속시간']

In [29]:
tems.shape

(9911403, 9)

In [30]:
tems.to_csv('./tems_vio_20_22.csv',index=False,encoding='cp949')

# 기상정보

In [40]:
weather = pd.read_csv('../../../011.데이터/09.ETC/OBS_ASOS_DD_20231019150617.csv',encoding='cp949')

In [41]:
weather.columns

Index(['지점', '지점명', '일시', '평균기온(°C)', '최저기온(°C)', '최저기온 시각(hhmi)', '최고기온(°C)',
       '최고기온 시각(hhmi)', '강수 계속시간(hr)', '10분 최다 강수량(mm)', '10분 최다강수량 시각(hhmi)',
       '1시간 최다강수량(mm)', '1시간 최다 강수량 시각(hhmi)', '일강수량(mm)', '최대 순간 풍속(m/s)',
       '최대 순간 풍속 풍향(16방위)', '최대 순간풍속 시각(hhmi)', '최대 풍속(m/s)', '최대 풍속 풍향(16방위)',
       '최대 풍속 시각(hhmi)', '평균 풍속(m/s)', '풍정합(100m)', '최다풍향(16방위)',
       '평균 이슬점온도(°C)', '최소 상대습도(%)', '최소 상대습도 시각(hhmi)', '평균 상대습도(%)',
       '평균 증기압(hPa)', '평균 현지기압(hPa)', '최고 해면기압(hPa)', '최고 해면기압 시각(hhmi)',
       '최저 해면기압(hPa)', '최저 해면기압 시각(hhmi)', '평균 해면기압(hPa)', '가조시간(hr)',
       '합계 일조시간(hr)', '1시간 최다일사 시각(hhmi)', '1시간 최다일사량(MJ/m2)', '합계 일사량(MJ/m2)',
       '일 최심신적설(cm)', '일 최심신적설 시각(hhmi)', '일 최심적설(cm)', '일 최심적설 시각(hhmi)',
       '합계 3시간 신적설(cm)', '평균 전운량(1/10)', '평균 중하층운량(1/10)', '기사',
       '안개 계속시간(hr)'],
      dtype='object')

In [42]:
weather = weather[['지점명','일시','평균기온(°C)', '최저기온(°C)','최고기온(°C)','일강수량(mm)', '최대 순간 풍속(m/s)','평균 풍속(m/s)','평균 상대습도(%)','평균 증기압(hPa)']]

In [44]:
weather.to_csv('./use_data/csv/gg_weather_20to22.csv',index=False,encoding='cp949')

# 주민등록인구

In [36]:
pops = pd.read_csv('../../../011.데이터/09.ETC/202001_202212_주민등록인구및세대현황_월간.csv',encoding='cp949')

In [37]:
ndf = pops.iloc[:,0:5]
ndf['기준연월'] = ndf.columns[1][:8]
ndf.columns = ['행정구역','total','home','male','female','기준연월']
for i in range(5,len(pops.columns),4):
    subset = pops.iloc[:,i:i+4]
    subset['행정구역'] = pops.iloc[:,0]
    subset['기준연월'] = subset.columns[0][:8]
    subset.columns = ['total','home','male','female','행정구역','기준연월']
    subset = subset[['행정구역','total','home','male','female','기준연월']]
    ndf = pd.concat([ndf,subset])

ndf.reset_index(drop=True,inplace=True)
ndf = ndf[ndf['행정구역'] != '경기도  (4100000000)']
ndf['행정구역'] = ndf['행정구역'].apply(lambda x : x.replace('경기도 경기도 ',''))

def remove_brackets(text):
    result = re.sub(r'\(.*\)', '', text)
    return result

ndf['행정구역'] = ndf['행정구역'].apply(lambda x : remove_brackets(x).strip())
ndf = ndf[~ndf['행정구역'].str.endswith('구')]
ndf['기준연월'] = pd.to_datetime(ndf['기준연월'], format='%Y년%m월')
ndf = ndf[['기준연월','행정구역', 'total', 'home', 'male', 'female',]]

In [38]:
ndf.head()

,기준연월,행정구역,total,home,male,female
1,2020-01-01,수원시,"1,193,894","499,238","601,097","592,797"
6,2020-01-01,성남시,"942,649","400,854","466,582","476,067"
10,2020-01-01,의정부시,"451,876","191,062","222,611","229,265"
11,2020-01-01,안양시,"565,352","222,143","279,857","285,495"
14,2020-01-01,부천시,"828,947","340,274","411,912","417,035"


In [39]:
ndf.to_csv('./use_data/csv/gg_pop_20to22.csv',index=False,encoding='cp949')

# 차량등록대수

In [104]:
path = '../../../011.데이터/09.ETC/차량등록/'
result_df = pd.DataFrame()

# 파일 순회
for file in os.listdir(path):
    file_ym = file.split('_')[:2]
    file_ym = ''.join(file_ym)
    file_path = os.path.join(path, file)
    
    # CSV 파일을 데이터프레임으로 읽기
    df = pd.read_excel(file_path,sheet_name=1,skiprows=2)  # 파일 읽기 설정에 따라 변경
    
    df = df[['시도','시군구','Unnamed: 21']]
    df.rename(columns = {'Unnamed: 21' : '차량등록대수'},inplace=True)
    df.drop(0,inplace=True)
    df['차량등록대수'] = df['차량등록대수'].astype('int')
    
    gg_start_idx = df[df['시도'] == '경기'].index.to_list()[0]
    gg_end_idx = df[df['시도'] == '강원'].index.to_list()[0] - 1 
    
    gg_df = df.loc[gg_start_idx : gg_end_idx,:]
    gg_df = gg_df[(gg_df['시군구'] != '계')]
    gg_df['시도'].fillna('경기',inplace=True)
    
    gg_df.reset_index(drop = True, inplace = True)
    
    def extract_gu_or_si_name(text):
        result = re.search(r'(\S+(시|군))', text)
        if result:
            return result.group(0)
        else:
            return text
            
    gg_df['시도'] = gg_df['시군구'].apply(lambda x : extract_gu_or_si_name(x))
    
    gg_df = gg_df.groupby(['시도'],as_index=False).agg({'차량등록대수':'sum'})
    gg_df['기준연월'] = file_ym
    gg_df['시도'] = gg_df['시도'].apply(lambda x: '양평군' if x == '양평' else (x[:3].strip() + '시'))
    gg_df = gg_df.groupby(['시도','기준연월'],as_index=False).sum()
    # 결과 데이터프레임에 추가
    result_df = pd.concat([result_df, gg_df])
    result_df

result_df.reset_index(drop=True,inplace=True)


In [106]:
tdf = result_df.copy()

In [107]:
tdf.to_csv('./use_data/csv/gg_car_reg_20to22.csv',index=False,encoding='cp949')

# ETC(SECTION_LINK)

In [34]:
ref = pd.read_csv('./구간과속카메라_구간_LINK_ID저장.csv')

In [5]:
#tmp = gpd.read_file('../../../011.데이터/08.LICENSE_ETC/NODELINKDATA/MOCT_LINK.shp')
tmp = gpd.read_file('./use_data/gis/MOCT_LINK.shp')

In [35]:
ref['LINK_ID'] = ref['LINK_ID'].astype('str')

slink = tmp[tmp['LINK_ID'].isin(ref['LINK_ID'].to_list())]

delist = ['2140027301','2140043000','2334282000','2330014600','2334315301','2334315601','2140957500','2320240300','2320128100','2320132300']

slink = slink[~slink['LINK_ID'].isin(delist)]

In [55]:
slink.to_file('./use_data/gis/SLTD_SECTION_LINK.shp',driver='ESRI Shapefile',encoding='cp949')

In [51]:
sltd = gpd.read_file('./use_data/gis/SLTD_SECTION_LINK_5179.shp')

In [53]:
sltd.to_file('./use_data/gis/SLTD_SECTION_LINK_5179.shp',driver='ESRI Shapefile',encoding='utf8')

In [ ]:
ref

In [64]:
test = pd.merge(sltd,ref,how='left',left_on='LINK_ID',right_on='LINK_ID')

In [68]:
test = test[['LINK_ID','장비번호','구간명','geometry']]

In [70]:
test.to_file('./use_data/gis/section_polygon/test.shp',driver='ESRI Shapefile',encoding='cp949')

# ALL JOIN
- df.loc[365,:]
- 365일이 설치일

In [48]:
#MTS
mts_pnt = pd.read_csv('./use_data/csv/mts_est_21_pnt.csv',encoding='cp949')
mts_pnt.loc[159,'설치시군구명'] = '화성시'
mts_pnt.loc[160,'설치시군구명'] = '성남시'

mts_section = pd.read_csv('./use_data/csv/mts_est_21_sec.csv',encoding='cp949')
mts_section.loc[0:2,'설치시군구명'] = '용인시'
mts_section.loc[3:5,'설치시군구명'] = '성남시'
mts_section.loc[6:7,'설치시군구명'] = '화성시'
mts_section.loc[8:9,'설치시군구명'] = '안산시'
mts_section.loc[14:16,'설치시군구명'] = '오산시'
mts_section.loc[17:19,'설치시군구명'] = '안성시'
mts_section['시종점여부'] = mts_section['설치장소'].apply(lambda x : '시점' if '시점' in x else '종점')

#TAAS
acc_light = pd.read_csv('./use_data/csv/taas_light_acc_sltd.csv',encoding='cp949')
acc_speed = pd.read_csv('./use_data/csv/taas_speed_acc_sltd.csv',encoding='cp949')
acc_section = pd.read_csv('./use_data/csv/taas_section_acc_sltd.csv',encoding='cp949')

#TEMS
tems = pd.read_csv('./use_data/csv/tems_vio_20_22.csv',encoding='cp949')

#Weather
weather = pd.read_csv('./use_data/csv/gg_weather_20to22.csv',encoding='cp949')

#pops 
pops = pd.read_csv('./use_data/csv/gg_pop_20to22.csv',encoding='cp949')

#car_regist
car_reg = pd.read_csv('./use_data/csv/gg_car_reg_20to22.csv',encoding='cp949')

In [49]:
pnt_sltd_cols = ['장비번호','단속형태','설치일자','설치시군구명','기상관측연결지역명']

#mts 장비 활용 컬럼 추출
mts_pnt = mts_pnt[pnt_sltd_cols]
mts_section = mts_section[['장비번호','구간그룹명','단속형태','설치일자','설치시군구명','기상관측연결지역명','시종점여부']]
mts_sct = mts_section.copy()
mts_sct.loc[sorted(mts_sct[mts_sct['구간그룹명'] == '오성면구간']['장비번호'].index.to_list()+ mts_sct[mts_sct['구간그룹명'] == '신대리구간']['장비번호'].index.to_list()),'구간그룹명'] = '오성면구간,신대리구간'


#section idx-group name dict
sec_raw = pd.read_csv('./use_data/csv/mts_section.csv')
tmp = sec_raw[sec_raw['구간그룹명'].isin(mts_sct['구간그룹명'])][['장비번호','구간그룹명']].drop_duplicates()
sec_equ_dict = {key : value for key, value in zip(tmp['구간그룹명'],tmp['장비번호'])}

#taas extract
pnt_idx = mts_pnt['장비번호'].unique()
sct_idx = mts_sct['구간그룹명'].unique()
acc_light = acc_light[acc_light['장비번호'].isin(pnt_idx)]
acc_speed = acc_speed[acc_speed['장비번호'].isin(pnt_idx)]
acc_section = acc_section[acc_section['구간그룹명'].isin(sct_idx)]

data = [
    ("가평군", 1322),("고양시", 1318), ("과천시", 1332), ("광명시", 1309), ("광주시", 1319), ("구리시", 1310), ("군포시", 1333), ("김포시", 1327),
    ("남양주시", 1334), ("동두천시", 1330), ("부천시", 1306), ("성남시", 1303), ("수원시", 1302), ("시흥시", 1316), ("안산시", 1307), ("안성시", 1326), ("안양시", 1305),
    ("양주시", 1311),("양평군", 1323),("여주시", 1313),("연천군", 1320), ("오산시", 1335), ("용인시", 1325), ("의왕시", 1336), ("의정부시", 1304), ("이천시", 1324),
    ("파주시", 1317),("평택시", 1308), ("포천시", 1321), ("하남시", 1337), ("화성시", 1315)
]

taas_sgg_dict = {key : value for value,key in data}

acc_section['발생지시군구코드'] = acc_section['발생지시군구코드'].apply(lambda x : taas_sgg_dict.get(x))  

#taas group by
lng_gb = acc_light.groupby(['장비번호','발생일자'],as_index=False).agg(
    {'발생지시군구코드' : 'count',
    '사망자수' : 'sum',
    '중상자수' : 'sum',
    '경상자수' : 'sum',
    '부상신고자수' : 'sum',
     '관련차량대수' : 'sum',
     '발생시각' : 'mean'
    }
).rename(columns = {'발생지시군구코드' : '사고건수'})
lng_gb['발생일자'] = pd.to_datetime(lng_gb['발생일자'])


spd_gb = acc_speed.groupby(['장비번호','발생일자'],as_index=False).agg(
    {'발생지시군구코드' : 'count',
    '사망자수' : 'sum',
    '중상자수' : 'sum',
    '경상자수' : 'sum',
    '부상신고자수' : 'sum',
     '관련차량대수' : 'sum',
     '발생시각' : 'mean'
    }
).rename(columns = {'발생지시군구코드' : '사고건수'})
spd_gb['발생일자'] = pd.to_datetime(spd_gb['발생일자'])


acc_section['tmp'] = 1
sct_gb = acc_section.groupby(['구간그룹명','발생일자'],as_index=False).agg(
    {'tmp' : 'count',
    '사망자수' : 'sum',
    '중상자수' : 'sum',
    '경상자수' : 'sum',
    '부상신고자수' : 'sum',
     '관련차량대수' : 'sum',
     '발생시각' : 'mean'
    }
).rename(columns = {'tmp' : '사고건수'})
sct_gb['발생일자'] = pd.to_datetime(sct_gb['발생일자'])

#tems group by
tems.drop(columns = ['단속발생시간','단속유형','단속시간'],inplace=True)
sltd_tems = tems.groupby(['장비번호','단속일자'],as_index=False).agg({
    '사건번호' : 'count',
    '제한속도' : 'mean',
    '과속속도' : 'mean',
    '초과속도' : 'mean'
}).rename(columns = {'사건번호' : '단속건수'})

sct_tems = pd.merge(tems,mts_sct[['장비번호','구간그룹명']].drop_duplicates(), left_on = ['장비번호'],right_on = ['장비번호'],how='left')
sct_tems = sct_tems[~sct_tems['구간그룹명'].isna()]

sct_tems_gb = sct_tems.groupby(['구간그룹명','단속일자'],as_index=False).agg({
    '장비번호' : 'count',
    '제한속도' : 'mean',
    '과속속도' : 'mean',
    '초과속도' : 'mean'
}).rename(columns = {'장비번호' : '단속건수'})


#결합용 데이터셋 날짜 변환
sltd_tems['단속일자'] = pd.to_datetime(sltd_tems['단속일자'])
sct_tems_gb['단속일자'] = pd.to_datetime(sct_tems_gb['단속일자'])
weather['일시'] = pd.to_datetime(weather['일시'])

pops['기준연월'] = pd.to_datetime(pops['기준연월'])
pops['기준연월'] = pops['기준연월'].dt.strftime('%Y년%m월')



## 지점단속장비 

In [50]:
df = mts_pnt[mts_pnt['장비번호'] == 'F9906']
df.reset_index(drop=True,inplace=True)

#기준일자 생성
std_date = df.iloc[0,2]
std_date = pd.Timestamp(std_date)

#기준일자 전후 365일 기간 일자 생성
date_range_col = pd.date_range(start=std_date - pd.DateOffset(days=365), end=std_date + pd.DateOffset(days=365),inclusive='left')


In [51]:
result = pd.DataFrame()
for equ_idx in tqdm(mts_pnt['장비번호'].unique()):
    #장비번호별 MTS 데이터 추출
    df = mts_pnt[mts_pnt['장비번호'] == equ_idx]
    df.reset_index(drop=True,inplace=True)
    
    #기준일자 생성
    std_date = df.iloc[0,2]
    std_date = pd.Timestamp(std_date)
    
    #기준일자 전후 365일 기간 일자 생성
    date_range_col = pd.date_range(std_date - pd.DateOffset(days=365), std_date + pd.DateOffset(days=365),inclusive='left')
    #결합용 dataframe 생성
    tdf = pd.DataFrame()
    tdf['date'] = date_range_col
    tdf['equ_idx'] = equ_idx
    tdf['vio_type'] = df.iloc[0,1]
    tdf['equ_est_region'] = df.iloc[0,3]
    tdf['weather_region'] = df.iloc[0,4]
    
    #공공데이터 결합용 날짜
    tdf['public_date_ym'] = tdf['date'].apply(lambda x : x.strftime('%Y년%m월'))
    
    #taas 결합
    if df.iloc[0,1] == '신호위반':
        taas_df = lng_gb[lng_gb['장비번호'] == equ_idx]
        tdf = pd.merge(tdf,taas_df, left_on = ['date','equ_idx'], right_on = ['발생일자','장비번호'],how='left')
        for i in taas_df.columns:
            tdf[i].fillna(0,inplace=True)
    elif df.iloc[0,1] == '속도위반':
        taas_df = spd_gb[spd_gb['장비번호'] == equ_idx]
        tdf = pd.merge(tdf,taas_df, left_on = ['date','equ_idx'], right_on = ['발생일자','장비번호'],how='left')
        for i in taas_df.columns:
            tdf[i].fillna(0,inplace=True)
            
    #TEMS 결합
    tems_df = sltd_tems[sltd_tems['장비번호'] == equ_idx]
    tdf = pd.merge(tdf,tems_df,left_on = ['date','equ_idx'], right_on =['단속일자','장비번호'],how='left')
    for i in ['단속건수','제한속도', '과속속도', '초과속도']:
        tdf[i].fillna(0, inplace=True)

    #날씨 결합
    weather_df = weather[weather['지점명'] == tdf.iloc[0,4]]
    tdf = pd.merge(tdf,weather_df, left_on=['date','weather_region'],right_on=['일시','지점명'],how='left')
    for i in ['평균기온(°C)', '최저기온(°C)', '최고기온(°C)', '일강수량(mm)','최대 순간 풍속(m/s)', '평균 풍속(m/s)', '평균 상대습도(%)', '평균 증기압(hPa)']:
        tdf[i].fillna(0,inplace=True)

    #인구 결합
    pops_df = pops[pops['행정구역'] == tdf.iloc[0,3]][['기준연월','행정구역','total']]
    tdf = pd.merge(tdf,pops_df, left_on = ['public_date_ym','equ_est_region'], right_on = ['기준연월','행정구역'],how='left')

    tdf['total'].fillna(0,inplace=True)

    #차량등록대수 결합
    car_reg_df = car_reg[car_reg['시도'] == tdf.iloc[0,3]]
    tdf = pd.merge(tdf, car_reg_df, left_on = ['public_date_ym','equ_est_region'], right_on = ['기준연월','시도'],how='left')
    tdf['차량등록대수'].fillna(0,inplace=True)

    # 불필요 컬럼 제거
    tdf.drop(columns = ['장비번호_x', '발생일자','장비번호_y', '단속일자', '지점명', '일시','기준연월_x', '행정구역','시도','기준연월_y'],inplace=True)
    tdf.rename(columns = {
        '평균기온(°C)' : '평균기온', 
        '최저기온(°C)' : '최저기온', 
        '최고기온(°C)' : '최고기온', 
        '일강수량(mm)' : '일강수량',
       '최대 순간 풍속(m/s)' : '최대순간풍속', 
        '평균 풍속(m/s)' : '평균풍속', 
        '평균 상대습도(%)' : '평균상대습도', 
        '평균 증기압(hPa)' : '평균증기압'
    }, inplace=True)
    
    #결과 병합
    result = pd.concat([result,tdf])
    
result.reset_index(drop=True,inplace=True)
result['total'] = result['total'].apply(lambda x : int(x.replace(',','')))
result.rename(columns = {'total' : 'equ_est_region_pops'},inplace=True)
result.columns = [
    '기준일자','장비번호','단속형태','설치지역','기상관측지역','기준연월','사고건수','사망자수','중상자수','경상자수','부상신고자수','관련차량대수',
    '평균발생시각','단속건수','평균제한속도','평균과속속도','평균초과속도',
    '일평균기온','일최저기온','일최고기온','일강수량','일최대순간풍속','일평균풍속','일평균상대습도','일평균중기압',
    '장비설치지역_주민등록인구수','장비설치지역_차량등록대수']

  0%|          | 0/423 [00:00<?, ?it/s]

In [52]:
result.to_csv('./use_data/result/input/PNT_EQU_RAW_daysum.csv',index=False, encoding='cp949')

## 구간단속장비
- 지점단속장비와 달리 장비번호로 JOIN을 먼저 해야 함
- 장비번호가 있는 데이터
    - TAAS / TEMS
      > 장비번호에 해당하는 TEMS와 TAAS를 결합
    - 시점과 종점의 행정구역이 다른경우는?
      > 시점의 행정구역으로 통일
    - 오성면과 신대리는 합쳐야 함
      > 오성면구간,신대리구간 으로 통일

In [56]:
result = pd.DataFrame()

for sct_nm in tqdm(mts_sct['구간그룹명'].unique()):
    #장비번호별 MTS 데이터 추출
    df = mts_sct[(mts_sct['구간그룹명'] == sct_nm)&(mts_sct['시종점여부']=='시점')]
    df.reset_index(drop=True,inplace=True)
    
    #기준일자 생성
    std_date = df.iloc[0,3]
    std_date = pd.Timestamp(std_date)
    
    #기준일자 전후 365일 기간 일자 생성
    date_range_col = pd.date_range(std_date - pd.DateOffset(days=365), std_date + pd.DateOffset(days=365),inclusive='left')

    #새로운 dataframe 생성 및 결합
    tdf = pd.DataFrame()
    tdf['date'] = date_range_col
    tdf['section_nm'] = df.iloc[0,1]
    tdf['vio_type'] = df.iloc[0,2]
    tdf['equ_est_region'] = df.iloc[0,4]
    tdf['weather_region'] = df.iloc[0,5]
    
    #공공데이터 결합용 날짜
    tdf['public_date_ym'] = tdf['date'].apply(lambda x : x.strftime('%Y년%m월'))

    #TEMS
    tdf = pd.merge(tdf,sct_tems_gb,left_on = ['date','section_nm'], right_on = ['단속일자','구간그룹명'],how='left')
    tdf.drop(columns = ['구간그룹명','단속일자'],inplace=True)
    for i in ['단속건수','제한속도','과속속도','초과속도']:
        tdf[i].fillna(0,inplace=True)

    #TAAS 결합
    tdf = pd.merge(tdf,sct_gb, left_on = ['date','section_nm'], right_on = ['발생일자','구간그룹명'],how='left')
    for i in sct_gb.drop(columns=['구간그룹명','발생일자']).columns:
        tdf[i].fillna(0,inplace=True)
    tdf.drop(columns = ['구간그룹명','발생일자'],inplace=True)
    
 
    #날씨 결합
    weather_df = weather[weather['지점명'] == tdf.iloc[0,4]]
    tdf = pd.merge(tdf,weather_df, left_on=['date','weather_region'],right_on=['일시','지점명'],how='left')
    for i in ['평균기온(°C)', '최저기온(°C)', '최고기온(°C)', '일강수량(mm)','최대 순간 풍속(m/s)', '평균 풍속(m/s)', '평균 상대습도(%)', '평균 증기압(hPa)']:
        tdf[i].fillna(0,inplace=True)

    #인구 결합
    pops_df = pops[pops['행정구역'] == tdf.iloc[0,3]][['기준연월','행정구역','total']]
    tdf = pd.merge(tdf,pops_df, left_on = ['public_date_ym','equ_est_region'], right_on = ['기준연월','행정구역'],how='left')
    tdf['total'].fillna(0,inplace=True)

    #차량등록대수 결합
    car_reg_df = car_reg[car_reg['시도'] == tdf.iloc[0,3]]
    tdf = pd.merge(tdf, car_reg_df, left_on = ['public_date_ym','equ_est_region'], right_on = ['기준연월','시도'],how='left')
    tdf['차량등록대수'].fillna(0,inplace=True)
    
    # 불필요 컬럼 제거
    tdf.drop(columns = ['지점명', '일시','기준연월_x', '행정구역','시도','기준연월_y'],inplace=True)
    tdf.rename(columns = {
        '평균기온(°C)' : '평균기온', 
        '최저기온(°C)' : '최저기온', 
        '최고기온(°C)' : '최고기온', 
        '일강수량(mm)' : '일강수량',
       '최대 순간 풍속(m/s)' : '최대순간풍속', 
        '평균 풍속(m/s)' : '평균풍속', 
        '평균 상대습도(%)' : '평균상대습도', 
        '평균 증기압(hPa)' : '평균증기압'
    }, inplace=True)
    result = pd.concat([result,tdf])
    
    
result.reset_index(drop=True,inplace=True)
result['total'] = result['total'].apply(lambda x : int(x.replace(',','')))
result.rename(columns = {'total' : 'equ_est_region_pops'},inplace=True)    
result.columns = [
    '기준일자','구간그룹명','단속형태','설치지역','기상관측지역','기준연월','단속건수','평균제한속도','평균과속속도','평균초과속도',
    '사고건수','사망자수','중상자수','경상자수','부상신고자수','관련차량대수',
    '평균발생시각','일평균기온','일최저기온','일최고기온','일강수량','일최대순간풍속','일평균풍속','일평균상대습도','일평균중기압',
    '장비설치지역_주민등록인구수','장비설치지역_차량등록대수'
]


  0%|          | 0/4 [00:00<?, ?it/s]

In [57]:
result.head()

,기준일자,구간그룹명,단속형태,설치지역,기상관측지역,기준연월,단속건수,평균제한속도,평균과속속도,평균초과속도,사고건수,사망자수,중상자수,경상자수,부상신고자수,관련차량대수,평균발생시각,일평균기온,일최저기온,일최고기온,일강수량,일최대순간풍속,일평균풍속,일평균상대습도,일평균중기압,장비설치지역_주민등록인구수,장비설치지역_차량등록대수
0,2020-05-27,경부고속도로390,구간과속,용인시,수원,2020년05월,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,17.1,12.0,24.1,0.0,9.9,2.3,77.0,14.4,1070987,470259
1,2020-05-28,경부고속도로390,구간과속,용인시,수원,2020년05월,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,18.0,12.5,23.8,0.0,7.5,1.7,73.9,14.6,1070987,470259
2,2020-05-29,경부고속도로390,구간과속,용인시,수원,2020년05월,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,19.4,13.1,27.4,0.0,7.3,1.6,74.1,16.0,1070987,470259
3,2020-05-30,경부고속도로390,구간과속,용인시,수원,2020년05월,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,21.6,15.1,29.7,0.0,8.4,1.6,64.8,15.8,1070987,470259
4,2020-05-31,경부고속도로390,구간과속,용인시,수원,2020년05월,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,20.4,15.8,27.6,1.2,8.2,2.0,71.8,16.6,1070987,470259


In [58]:
result.to_csv('./use_data/result/input/SCT_EQU_RAW_daysum.csv',index=False,encoding='cp949')